# Rule: **build_industrial_distribution_key**


**Description**

This rule uses the Hotmaps and Global Energy Monitor (GEM) database. First it removes entries without valid locations. Once all the industrial plants have valid coordenates, they are assigned to a bus region based on its location. Finally, the rule calculates the nodal distribution key for each sector based on Hotmaps ETS/EPRTR emissions of the industrial sites in each region, with exceptions for:
- Cement: clinker capacity from GEM (directly obtained or estimated by cement capacity)
- Iron and Steel: technology distribution from GEM (EAF, DRI + EAF, integrated steelworks)
- Ammonia:  production per industrial plant
- Refineries: production capacity

This leads to a distribution key for every sector and bus of 0-1, and the sum over buses of one sector per country is 1. The following subcategories of industry are considered:
- Iron and steel
- Cement
- Refineries
- Paper and printing
- Chemical industry
- Glass
- Non-ferrous metals
- Non-metallic mineral products
- Other non-classified  

Additionally, the population distribution is added and it is used when there are no industries for a particular subcategory.

**Inputs**

- resources/{prefix}/{name}/`regions_onshore_base_s_{clusters}.geojson`
- data/hotmaps_industrial_sites/archive/version-1.1/`Industrial_database.csv`
- data/gem_gspt/archive/march-2025-v1.1/`Global-Steel-Plant-Tracker.xlsx`
- data/gem_gcct/archive/july-2025/`Global-Cement-and-Concrete-Tracker.xlsx`
- data/`ammonia_plants.csv`
- data/`refineries-noneu.csv`
- resources/{prefix}/{name}/`clustered_pop_layout.csv`


**Outputs**

- resources/{prefix}/{name}/`industrial_distribution_key_base_s_{clusters}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
clusters = '' # number of clusters or 'adm'

In [ ]:
##### Imports
import pandas as pd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import os 
import sys
from matplotlib.colors import Normalize

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Region files
region_tag = f"base_s_{clusters}"
gdf_regions_onshore, gdf_regions_offshore = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=region_tag,
)

##### Set options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## `industrial_distribution_key_base_s_{clusters}.csv`  
Load the file and preview its content.

In [ ]:
file = f"industrial_distribution_key_base_s_" f"{clusters}"".csv"

distr_key = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

distr_key

How are these industrial distribution keys spatially distributed across subsectors?

In [ ]:
#################### Parameters 

### Industrial sectors to plot. Choose any subset, in desired order.
sector_variables = [
    'Cement',
    'Paper and printing',    
    #'Refineries',
    #'Iron and steel',
    #'Glass',
    #'Chemical industry',
    #'Non-metallic mineral products',
    #'Non-ferrous metals',
    #'EAF',
    #'DRI + EAF',
    #'Integrated steelworks',
    #'Ammonia',
    #'population'
]

### Plotting parameters
font_size = 15

#################### Plot

# Merge with regions
gdf = gdf_regions_onshore.merge(
    distr_key,
    left_on="name",
    right_on=distr_key.columns[0],
    how="left"
)

# Common color scale
vmin = 0
vmax = 1
norm = Normalize(vmin=vmin, vmax=vmax)

# Layout: 3 columns, rows determined by the number of selected variables
n_vars = len(sector_variables)
n_cols = min(3, n_vars)
n_rows = int(np.ceil(n_vars / n_cols))

fig_size = [6 * n_cols, 5 * n_rows]
crs = ccrs.PlateCarree()

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=fig_size,
    constrained_layout=True,
    subplot_kw={'projection': crs},
)

# Flatten axes for easy iteration (works for 1, 2 or more subplots)
axes_flat = np.atleast_1d(axes).flatten()

for ax, var in zip(axes_flat, sector_variables):
    gdf.plot(
        column=var,
        cmap="RdBu",
        norm=Normalize(vmin=0, vmax=1),
        linewidth=0.8,
        ax=ax,
        edgecolor="black",
        legend=False
    )
    total = gdf[var].sum()
    ax.set_title(f"{var.capitalize()}", fontsize=font_size)


    ### Add map features
    xp.map_add_features(ax, params['map_add_features'])

# Hide any unused subplots
for ax in axes_flat[n_vars:]:
    ax.set_visible(False)

# Shared colorbar (aligned with the visible subplots)
sm = plt.cm.ScalarMappable(cmap="RdBu", norm=norm)
sm._A = []

cbar = fig.colorbar(
    sm,
    ax=axes_flat[:n_vars].tolist(),
    location="right",
    shrink=0.8
)
cbar.set_label("Distribution key (0-1)", fontsize=font_size)
cbar.ax.tick_params(labelsize=font_size)

fig.suptitle(
    "Industrial distribution keys",
    fontsize=font_size*1.2
)